In [1]:
!pip install numpy==1.26.4 -q
!pip install nuscenes-devkit -q
!pip install segmentation-models-pytorch -q
!pip install albumentations -q
!pip install torchvision -q  # needed for deform_conv2d
# RESTART RUNTIME after this

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, tarfile

TGZ_PATH   = '/content/drive/MyDrive/Copy of v1.0-mini.tgz'
EXTRACT_TO = '/content/nuscenes'
os.makedirs(EXTRACT_TO, exist_ok=True)

if not os.path.exists('/content/nuscenes/v1.0-mini'):
    print("Extracting...")
    with tarfile.open(TGZ_PATH, 'r:gz') as tar:
        tar.extractall(EXTRACT_TO)
    print("Done!")
else:
    print("Already extracted.")

In [5]:
import os, cv2, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import LidarPointCloud
import segmentation_models_pytorch as smp
import albumentations as A
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

torch.manual_seed(42)           # fixes random variation between runs
np.random.seed(42)

DEVICE            = 'cuda' if torch.cuda.is_available() else 'cpu'
NUSCENES_DATAROOT = '/content/nuscenes'
NUSCENES_VERSION  = 'v1.0-mini'
IMG_H, IMG_W      = 900, 1600
K  = np.array([[1266.417,0,816.267],[0,1266.417,491.503],[0,0,1]])
CAM_HEIGHT        = 1.51
CAMERAS           = ['CAM_FRONT','CAM_FRONT_LEFT','CAM_FRONT_RIGHT',
                     'CAM_BACK','CAM_BACK_LEFT','CAM_BACK_RIGHT']
print(f"Device: {DEVICE}")

nusc = NuScenes(version=NUSCENES_VERSION,
                dataroot=NUSCENES_DATAROOT, verbose=False)
print(f"Loaded {len(nusc.sample)} samples")

# ── Fix random split ──────────────────────────────────────────────
# Sort by scene so train/val split is deterministic and scene-balanced
all_idx = list(range(len(nusc.sample)))
split   = int(0.8*len(all_idx))
train_idx = all_idx[:split]
val_idx   = all_idx[split:]

# ── Distance weight map ───────────────────────────────────────────
def build_dist_weight():
    x,y   = np.linspace(-20,20,200),np.linspace(-20,20,200)
    xx,yy = np.meshgrid(x,y)
    d     = np.sqrt(xx**2+yy**2).astype(np.float32)
    w     = 1.0/np.clip(d,1.0,None); w=w/w.mean()
    return torch.from_numpy(w).unsqueeze(0).unsqueeze(0).to(DEVICE)

DIST_WEIGHT = build_dist_weight()

# ── IPM Homography ────────────────────────────────────────────────
def build_H(cam_name):
    fx,fy,cx,cy=K[0,0],K[1,1],K[0,2],K[1,2]
    yaw_deg={'CAM_FRONT':0,'CAM_FRONT_LEFT':55,'CAM_FRONT_RIGHT':-55,
             'CAM_BACK':180,'CAM_BACK_LEFT':125,'CAM_BACK_RIGHT':-125}
    yaw=np.radians(yaw_deg[cam_name]); cy_,sy_=np.cos(yaw),np.sin(yaw)
    wp=np.float32([[-20,-20,0],[-20,20,0],[20,20,0],[20,-20,0]])
    def w2i(p):
        X=p[0]*cy_+p[1]*sy_; Y=-p[0]*sy_+p[1]*cy_
        X=max(X,0.5); return [cx+fx*(Y/X),cy-fy*(CAM_HEIGHT/X)]
    def w2b(p): return [(p[0]+20)/0.2,(p[1]+20)/0.2]
    try:
        H,_=cv2.findHomography(np.float32([w2i(p) for p in wp]),
                                np.float32([w2b(p) for p in wp])); return H
    except: return None

IPM_Hs={c:build_H(c) for c in CAMERAS}

# ── Data helpers ──────────────────────────────────────────────────
def get_bev(nusc, idx):
    fused=np.zeros((200,200,3),dtype=np.float32)
    count=np.zeros((200,200,1),dtype=np.float32)
    for cam in CAMERAS:
        H=IPM_Hs.get(cam)
        if H is None: continue
        try:
            sd =nusc.get('sample_data',nusc.sample[idx]['data'][cam])
            img=cv2.cvtColor(cv2.imread(
                    os.path.join(NUSCENES_DATAROOT,sd['filename'])),
                    cv2.COLOR_BGR2RGB).astype(np.float32)
            bev=cv2.warpPerspective(img,H,(200,200),flags=cv2.INTER_LINEAR,
                                     borderMode=cv2.BORDER_CONSTANT)
            mask=(bev.sum(2)>0).astype(np.float32)[:,:,None]
            fused+=bev*mask; count+=mask
        except: continue
    return (fused/np.maximum(count,1)).astype(np.uint8)

def get_gt(nusc, idx):
    sd =nusc.get('sample_data',nusc.sample[idx]['data']['LIDAR_TOP'])
    pc =LidarPointCloud.from_file(os.path.join(NUSCENES_DATAROOT,sd['filename']))
    pts=pc.points[:3].T
    occ=np.zeros((200,200),dtype=np.float32)
    m  =((pts[:,0]>=-20)&(pts[:,0]<20)&(pts[:,1]>=-20)&(pts[:,1]<20)&
         (pts[:,2]>=0.2)&(pts[:,2]<2.5))
    p=pts[m]
    if len(p):
        px=np.clip(((p[:,0]+20)/0.2).astype(int),0,199)
        py=np.clip(((p[:,1]+20)/0.2).astype(int),0,199)
        occ[py,px]=1.0
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5))
    return cv2.dilate(occ,k,iterations=2)

def get_future_gt(nusc, idx, n_future=6):
    """Req 2: accumulate next ~3 seconds of LiDAR (6 frames @ 2Hz)"""
    sample =nusc.sample[idx]
    future =np.zeros((200,200),dtype=np.float32)
    cur    =sample
    count  =0
    for _ in range(n_future):
        if cur['next']=='': break
        cur=nusc.get('sample',cur['next'])
        try:
            sd =nusc.get('sample_data',cur['data']['LIDAR_TOP'])
            pc =LidarPointCloud.from_file(
                    os.path.join(NUSCENES_DATAROOT,sd['filename']))
            pts=pc.points[:3].T
            m  =((pts[:,0]>=-20)&(pts[:,0]<20)&(pts[:,1]>=-20)&(pts[:,1]<20)&
                 (pts[:,2]>=0.2)&(pts[:,2]<2.5))
            p=pts[m]
            if len(p):
                px=np.clip(((p[:,0]+20)/0.2).astype(int),0,199)
                py=np.clip(((p[:,1]+20)/0.2).astype(int),0,199)
                future[py,px]+=1.0
            count+=1
        except: continue
    if count>0: future=np.clip(future/count,0,1)
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5))
    return cv2.dilate(future,k,iterations=1)

def get_front(nusc, idx):
    sd =nusc.get('sample_data',nusc.sample[idx]['data']['CAM_FRONT'])
    img=cv2.cvtColor(cv2.imread(os.path.join(NUSCENES_DATAROOT,sd['filename'])),
                     cv2.COLOR_BGR2RGB)
    return cv2.resize(img,(400,224))

def get_pv_label(nusc, idx):
    sd =nusc.get('sample_data',nusc.sample[idx]['data']['LIDAR_TOP'])
    pc =LidarPointCloud.from_file(os.path.join(NUSCENES_DATAROOT,sd['filename']))
    pts=pc.points[:3].T
    lbl=np.zeros((IMG_H,IMG_W),dtype=np.float32)
    m  =pts[:,0]>0.5; p=pts[m]
    u  =(K[0,0]*p[:,1]/p[:,0]+K[0,2]).astype(int)
    v  =(K[1,1]*(-p[:,2]+CAM_HEIGHT)/p[:,0]+K[1,2]).astype(int)
    ok =(u>=0)&(u<IMG_W)&(v>=0)&(v<IMG_H)&(p[:,2]>-0.5)&(p[:,2]<2.5)
    lbl[v[ok],u[ok]]=1.0
    return cv2.resize(lbl,(400,224))

aug=A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.HueSaturationValue(p=0.4),
    A.GaussNoise(p=0.3),
    A.Rotate(limit=15,p=0.4),
    A.GridDistortion(p=0.2),
    A.RandomGamma(p=0.3),
    A.CLAHE(p=0.2),
],additional_targets={'mask':'mask','mask2':'mask'})

class BEVDataset(Dataset):
    def __init__(self,nusc,indices,augment=False):
        self.augment=augment; self.cache={}
        print(f"Caching {len(indices)} samples...")
        for i,idx in enumerate(indices):
            try:
                bev   =get_bev(nusc,idx)
                gt    =get_gt(nusc,idx)
                future=get_future_gt(nusc,idx)
                fr    =get_front(nusc,idx)
                pv    =get_pv_label(nusc,idx)
                self.cache[idx]=(bev,gt,future,fr,pv)
            except Exception as e: print(f"  skip {idx}: {e}")
            if i%80==0: print(f"  {i}/{len(indices)}")
        self.indices=[i for i in indices if i in self.cache]
        print(f"Cached {len(self.indices)} OK")

    def __len__(self): return len(self.indices)

    def __getitem__(self,i):
        bev,gt,future,fr,pv=self.cache[self.indices[i]]
        if self.augment:
            r=aug(image=bev,mask=gt,mask2=future)
            bev,gt,future=r['image'],r['mask'],r['mask2']
        bev   =torch.from_numpy(bev.astype(np.float32)/255.).permute(2,0,1)
        gt    =torch.from_numpy(gt.astype(np.float32)).unsqueeze(0)
        future=torch.from_numpy(future.astype(np.float32)).unsqueeze(0)
        fr    =torch.from_numpy(fr.astype(np.float32)/255.).permute(2,0,1)
        pv    =torch.from_numpy(pv.astype(np.float32)).unsqueeze(0)
        return bev,gt,future,fr,pv

train_ds=BEVDataset(nusc,train_idx,augment=True)
val_ds  =BEVDataset(nusc,val_idx,  augment=False)
train_dl=DataLoader(train_ds,batch_size=4,shuffle=True, num_workers=0,
                     worker_init_fn=lambda _:np.random.seed(42))
val_dl  =DataLoader(val_ds,  batch_size=4,shuffle=False,num_workers=0)
print(f"Train:{len(train_ds)}  Val:{len(val_ds)}")

# Sanity check
bev_s,gt_s,fut_s,_,_=train_ds[0]
print(f"GT occupied: {100*gt_s.mean():.1f}%  Future: {100*fut_s.mean():.1f}%")
assert gt_s.mean()>0.01, "GT empty!"

# ── Model: 3 outputs for all 3 hackathon requirements ────────────
class PVHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.e1=nn.Sequential(nn.Conv2d(3, 32,3,stride=2,padding=1,bias=False),nn.BatchNorm2d(32),nn.ReLU())
        self.e2=nn.Sequential(nn.Conv2d(32,64,3,stride=2,padding=1,bias=False),nn.BatchNorm2d(64),nn.ReLU())
        self.e3=nn.Sequential(nn.Conv2d(64,128,3,stride=2,padding=1,bias=False),nn.BatchNorm2d(128),nn.ReLU())
        self.l1=nn.Conv2d(128,32,1); self.l2=nn.Conv2d(64,32,1); self.l3=nn.Conv2d(32,32,1)
        self.seg=nn.Conv2d(32,1,1)

    def forward(self,x):
        e1=self.e1(x); e2=self.e2(e1); e3=self.e3(e2)
        d=F.interpolate(self.l1(e3),size=e2.shape[2:],mode='bilinear',align_corners=False)
        d=F.interpolate(d+self.l2(e2),size=e1.shape[2:],mode='bilinear',align_corners=False)
        d=F.interpolate(d+self.l3(e1),size=x.shape[2:],mode='bilinear',align_corners=False)
        return self.seg(d)

class BEVOccNet(nn.Module):
    """
    Input : (B,3,200,200) — 6-camera fused BEV
    Output:
      current (B,1,200,200) — Req 1: current occupancy grid
      future  (B,1,200,200) — Req 2: future BEV next 3s
      uncert  (B,1,200,200) — Req 3: uncertainty / risk heatmap
    """
    def __init__(self):
        super().__init__()
        # Shared backbone — pretrained EfficientNet-b4
        self.unet=smp.Unet(
            encoder_name    ='efficientnet-b4',
            encoder_weights ='imagenet',
            in_channels     =3,
            classes         =32,           # shared feature map
            activation      =None,
            decoder_channels=(256,128,64,32,32))

        # Req 1: current occupancy
        self.head_curr=nn.Sequential(
            nn.Conv2d(32,16,3,padding=1,bias=False),
            nn.BatchNorm2d(16),nn.ReLU(inplace=True),
            nn.Conv2d(16,1,1))

        # Req 2: future BEV (next 3 seconds)
        self.head_future=nn.Sequential(
            nn.Conv2d(32,16,3,padding=1,bias=False),
            nn.BatchNorm2d(16),nn.ReLU(inplace=True),
            nn.Conv2d(16,1,1))

        # Req 3: uncertainty heatmap
        self.head_uncert=nn.Sequential(
            nn.Conv2d(32,16,3,padding=1,bias=False),
            nn.BatchNorm2d(16),nn.ReLU(inplace=True),
            nn.Conv2d(16,1,1))

        # PV supervision head
        self.pv_head=PVHead()

        # Bias init: sigmoid(-2.94) ≈ 0.05
        for h in [self.head_curr,self.head_future]:
            nn.init.constant_(h[-1].bias,-2.94)
        nn.init.constant_(self.head_uncert[-1].bias,0.0)

    def forward(self,bev,front=None):
        feat  =self.unet(bev)                  # (B,32,200,200)
        curr  =self.head_curr(feat)             # Req 1
        future=self.head_future(feat)           # Req 2
        uncert=self.head_uncert(feat)           # Req 3
        pv    =self.pv_head(front) if (front is not None
                                        and self.training) else None
        return curr,future,uncert,pv

model=BEVOccNet().to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Shape check
with torch.no_grad():
    d=torch.zeros(2,3,200,200).to(DEVICE)
    c,f,u,_=model(d)
    print(f"Req1 current : {c.shape}")
    print(f"Req2 future  : {f.shape}")
    print(f"Req3 uncert  : {u.shape}")

# ── Loss ──────────────────────────────────────────────────────────
class FullLoss(nn.Module):
    def __init__(self): super().__init__()

    def focal(self,logits,targets,weight=None,pw=15.0):
        bce=F.binary_cross_entropy_with_logits(
            logits,targets,reduction='none',
            pos_weight=torch.tensor([pw]).to(logits.device))
        pt =(targets*torch.sigmoid(logits)+
             (1-targets)*(1-torch.sigmoid(logits)))
        fl =(0.75*targets+0.25*(1-targets))*(1-pt)**2.0*bce
        if weight is not None: fl=fl*weight
        return fl.mean()

    def dice(self,logits,targets):
        p=torch.sigmoid(logits); s=1e-6
        i=(p*targets).sum(dim=(2,3))
        return (1-(2*i+s)/(p.sum(dim=(2,3))+targets.sum(dim=(2,3))+s)).mean()

    def tversky(self,logits,targets):
        p=torch.sigmoid(logits); s=1e-6
        tp=(p*targets).sum(dim=(2,3))
        fp=(p*(1-targets)).sum(dim=(2,3))
        fn=((1-p)*targets).sum(dim=(2,3))
        return (1-(tp+s)/(tp+0.3*fp+0.7*fn+s)).mean()

    def occ_loss(self,logits,gt):
        dw=DIST_WEIGHT.expand(logits.shape[0],-1,-1,-1)
        return (0.25*self.focal(logits,gt)+
                0.25*self.focal(logits,gt,weight=dw)+
                0.25*self.dice(logits,gt)+
                0.25*self.tversky(logits,gt))

    def forward(self,curr,future,uncert,pv_out,gt,gt_f,pv_gt):
        lc=self.occ_loss(curr,gt)
        lf=self.occ_loss(future,gt_f)
        # Uncertainty: learn to predict where current prediction is wrong
        with torch.no_grad():
            err=(torch.sigmoid(curr)-gt).abs()
        lu=F.mse_loss(torch.sigmoid(uncert),err)
        lp=torch.tensor(0.,device=curr.device)
        if pv_out is not None:
            pv_r=F.interpolate(pv_gt,size=pv_out.shape[2:],mode='nearest')
            lp  =self.focal(pv_out,pv_r,pw=10.0)
        total=lc+0.5*lf+0.3*lu+0.3*lp
        return total,lc.item(),lf.item(),lu.item()

criterion=FullLoss()

# ── Metrics ───────────────────────────────────────────────────────
def calc_iou(logits,gt,t=0.35):
    p=(torch.sigmoid(logits)>=t); g=(gt>=0.5)
    return ((p&g).sum()/((p|g).sum()+1e-8)).item()

def calc_dwe(logits,gt):
    pred=torch.sigmoid(logits)
    x=np.clip(np.linspace(-20,20,200),0.5,None)
    w=torch.tensor(1./x,dtype=torch.float32).to(pred.device); w/=w.sum()
    p=pred.clamp(1e-7,1-1e-7)
    return (-(gt*torch.log(p)+(1-gt)*torch.log(1-p))*w).sum().item()

def tta_pred(model,bev):
    model.eval()
    with torch.no_grad():
        c1,f1,u1,_=model(bev)
        c2,f2,u2,_=model(torch.flip(bev,[3]))
        c2=torch.flip(c2,[3]); f2=torch.flip(f2,[3]); u2=torch.flip(u2,[3])
        c3,f3,u3,_=model(torch.flip(bev,[2]))
        c3=torch.flip(c3,[2]); f3=torch.flip(f3,[2]); u3=torch.flip(u3,[2])
    curr  =(torch.sigmoid(c1)+torch.sigmoid(c2)+torch.sigmoid(c3))/3.
    future=(torch.sigmoid(f1)+torch.sigmoid(f2)+torch.sigmoid(f3))/3.
    uncert=(torch.sigmoid(u1)+torch.sigmoid(u2)+torch.sigmoid(u3))/3.
    return curr,future,uncert

# ── Training ──────────────────────────────────────────────────────
best_iou_val=0; ACCUM=2

def run_epoch(use_tta=False):
    global best_iou_val
    model.train(); tl=tc=tf=tu=0
    optimizer.zero_grad()
    for step,(bev,gt,gt_f,fr,pv) in enumerate(train_dl):
        bev,gt,gt_f,fr,pv=(bev.to(DEVICE),gt.to(DEVICE),gt_f.to(DEVICE),
                            fr.to(DEVICE),pv.to(DEVICE))
        curr,future,uncert,pv_out=model(bev,fr)
        loss,lc,lf,lu=criterion(curr,future,uncert,pv_out,gt,gt_f,pv)
        (loss/ACCUM).backward()
        tl+=loss.item(); tc+=lc; tf+=lf; tu+=lu
        if (step+1)%ACCUM==0:
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            optimizer.step(); optimizer.zero_grad()

    model.eval(); vi=vd=0
    with torch.no_grad():
        for bev,gt,gt_f,fr,pv in val_dl:
            bev,gt=bev.to(DEVICE),gt.to(DEVICE)
            if use_tta:
                curr,_,_=tta_pred(model,bev)
                lg=torch.log(curr.clamp(1e-7,1-1e-7)/(1-curr).clamp(1e-7,1-1e-7))
                vi+=calc_iou(lg,gt); vd+=calc_dwe(lg,gt)
            else:
                c,_,_,_=model(bev); vi+=calc_iou(c,gt); vd+=calc_dwe(c,gt)

    n=len(train_dl); nv=len(val_dl)
    tl/=n;tc/=n;tf/=n;tu/=n;vi/=nv;vd/=nv
    scheduler.step()
    saved=""
    if vi>best_iou_val:
        best_iou_val=vi
        torch.save(model.state_dict(),'bev_final.pth')
        saved="  ✓"
    print(f"  Loss={tl:.4f} Curr={tc:.4f} Fut={tf:.4f} "
          f"Unc={tu:.4f} IoU={vi:.4f} DWE={vd:.4f}{saved}")
    return vi

# Phase 1
print("="*60+"\nPHASE 1 — Encoder frozen (10 epochs)\n"+"="*60)
for p in model.unet.encoder.parameters(): p.requires_grad=False
optimizer=torch.optim.AdamW(
    filter(lambda p:p.requires_grad,model.parameters()),
    lr=1e-3,weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.OneCycleLR(
    optimizer,max_lr=1e-3,
    steps_per_epoch=len(train_dl),
    epochs=10,pct_start=0.3)
for e in range(20):
    print(f"Epoch {e+1:02d}/10",end=" "); run_epoch()

# Phase 2
print("="*60+"\nPHASE 2 — Full fine-tune (30 epochs)\n"+"="*60)
for p in model.unet.encoder.parameters(): p.requires_grad=True
optimizer=torch.optim.AdamW([
    {'params':model.unet.encoder.parameters(),            'lr':5e-5},
    {'params':model.unet.decoder.parameters(),            'lr':2e-4},
    {'params':model.unet.segmentation_head.parameters(),  'lr':2e-4},
    {'params':model.head_curr.parameters(),               'lr':2e-4},
    {'params':model.head_future.parameters(),             'lr':2e-4},
    {'params':model.head_uncert.parameters(),             'lr':2e-4},
    {'params':model.pv_head.parameters(),                 'lr':2e-4},
],weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,T_0=10,T_mult=2,eta_min=1e-7)
no_imp=0
for e in range(60):
    print(f"Epoch {e+1:02d}/30",end=" ")
    v=run_epoch(use_tta=(e>=20))
    no_imp=0 if v>best_iou_val-0.001 else no_imp+1
    if no_imp>=15: print("Early stop."); break

print(f"\nBest IoU: {best_iou_val:.4f}")

# ── Final output: ALL 3 hackathon requirements ────────────────────
def render_full_output(curr_prob,future_prob,uncert_prob,
                        gt,bev_rgb,idx,iou,dwe,save_path=None):
    pred_bin=(curr_prob>0.5); gt_bin=(gt>0.5)
    overlay =np.zeros((200,200,3),dtype=np.uint8)
    overlay[pred_bin& gt_bin] =[0,  210, 90]
    overlay[pred_bin&~gt_bin] =[220, 50, 50]
    overlay[~pred_bin&gt_bin] =[50, 100,220]
    overlay[(~pred_bin&~gt_bin)]=(
        bev_rgb[(~pred_bin&~gt_bin)].astype(float)*0.25).astype(np.uint8)
    # Risk = high occupancy AND high uncertainty
    risk=(curr_prob*uncert_prob)
    risk=(risk-risk.min())/(risk.max()-risk.min()+1e-8)

    fig=plt.figure(figsize=(24,10),facecolor='#08080f')
    fig.suptitle(
        f'BEV Occupancy — Sample {idx} | IoU={iou:.4f} | DWE={dwe:.4f}',
        color='white',fontsize=14,fontweight='bold',y=0.99)

    panels=[
        (bev_rgb,     None,    'BEV Camera Input\n(6-camera fused)',    ''),
        (curr_prob,   'RdYlGn','Req 1: Current Occupancy Grid',         '✓ Req 1'),
        (overlay,     None,    'Req 1: TP / FP / FN Overlay',           '✓ Req 1'),
        (future_prob, 'hot',   'Req 2: Future BEV (next 3s)',           '✓ Req 2'),
        (uncert_prob, 'plasma','Req 3: Uncertainty Heatmap',            '✓ Req 3'),
        (risk,        'Reds',  'Req 3: Risk Zones\n(occupancy×uncert)', '✓ Req 3'),
    ]

    for k,(img,cmap,title,req) in enumerate(panels):
        ax=fig.add_subplot(2,3,k+1)
        ax.set_facecolor('#08080f')
        if img.ndim==3:
            im=ax.imshow(img,origin='lower',extent=[-20,20,-20,20])
        else:
            im=ax.imshow(img,cmap=cmap,origin='lower',
                          extent=[-20,20,-20,20],vmin=0,vmax=1)
            cb=plt.colorbar(im,ax=ax,fraction=0.046,pad=0.04)
            cb.ax.tick_params(colors='white',labelsize=6)
            cb.outline.set_edgecolor('#555')
            plt.setp(cb.ax.yaxis.get_ticklabels(),color='white')

        # Range rings
        for r in [5,10,15,20]:
            ax.add_patch(plt.Circle((0,0),r,fill=False,
                                     color='white',alpha=0.1,lw=0.6,ls='--'))
            if r<20: ax.text(r+0.3,0.3,f'{r}m',color='white',
                              fontsize=5,alpha=0.35)
        # Grid
        for v in range(-15,20,5):
            ax.axvline(v,color='white',alpha=0.04,lw=0.4)
            ax.axhline(v,color='white',alpha=0.04,lw=0.4)
        # Compass
        for pos,lbl in [((0,19.2),'N'),((0,-19.5),'S'),
                         ((19.2,0),'E'),((-19.5,0),'W')]:
            ax.text(pos[0],pos[1],lbl,color='cyan',fontsize=7,
                     ha='center',va='center',fontweight='bold',alpha=0.7)
        # Ego vehicle
        ax.add_patch(plt.Rectangle((-1,-2.2),2,4.4,lw=1.5,
                                    edgecolor='yellow',facecolor='yellow',
                                    alpha=0.85,zorder=5))
        ax.set_xlim(-20,20); ax.set_ylim(-20,20)
        ax.set_title(f'{title}\n{req}',color='white',fontsize=9,pad=3)
        ax.tick_params(colors='white',labelsize=6)
        for sp in ax.spines.values(): sp.set_edgecolor('#333')

        if 'TP' in title:
            ax.legend(handles=[
                mpatches.Patch(color='#00D25A',label='True Positive'),
                mpatches.Patch(color='#DC3232',label='False Positive'),
                mpatches.Patch(color='#3264DC',label='False Negative'),
            ],loc='lower right',fontsize=7,facecolor='#08080f',
              labelcolor='white',framealpha=0.7)

    plt.tight_layout(rect=[0,0,1,0.97])
    if save_path:
        plt.savefig(save_path,dpi=150,bbox_inches='tight',facecolor='#08080f')
        print(f"Saved: {save_path}")
    plt.show(); plt.close()

# ── Evaluate ──────────────────────────────────────────────────────
print("\nGenerating outputs for all 3 requirements...")
model.load_state_dict(torch.load('bev_final.pth'))
model.eval()
all_ious=[]; all_dwes=[]

with torch.no_grad():
    for i,(bev,gt,gt_f,fr,pv) in enumerate(val_dl):
        bev,gt=bev.to(DEVICE),gt.to(DEVICE)
        curr,future,uncert=tta_pred(model,bev)
        for j in range(bev.shape[0]):
            lg=torch.log(curr[j:j+1].clamp(1e-7,1-1e-7)/
                          (1-curr[j:j+1]).clamp(1e-7,1-1e-7))
            all_ious.append(calc_iou(lg,gt[j:j+1]))
            all_dwes.append(calc_dwe(lg,gt[j:j+1]))
        if i<3:
            for j in range(bev.shape[0]):
                render_full_output(
                    curr_prob  =curr[j,0].cpu().numpy(),
                    future_prob=future[j,0].cpu().numpy(),
                    uncert_prob=uncert[j,0].cpu().numpy(),
                    gt         =gt[j,0].cpu().numpy(),
                    bev_rgb    =(bev[j].cpu().permute(1,2,0).numpy()*255).astype(np.uint8),
                    idx        =i*4+j,
                    iou        =all_ious[-1],
                    dwe        =all_dwes[-1],
                    save_path  =f'submission_{i*4+j}.png')

print(f"\n{'='*55}")
print(f"  SUBMISSION CHECKLIST:")
print(f"  ✓ Req 1 — Current BEV Occupancy Grid     (IoU metric)")
print(f"  ✓ Req 2 — Future BEV next 3s             (6 LiDAR frames)")
print(f"  ✓ Req 3 — Uncertainty + Risk Heatmap")
print(f"  ✓ Distance-weighted Error metric")
print(f"  ✓ Deterministic (seed=42, fixed split)")
print(f"")
print(f"  Mean IoU (TTA) : {np.mean(all_ious):.4f} ± {np.std(all_ious):.4f}")
print(f"  Mean DWE       : {np.mean(all_dwes):.4f} ± {np.std(all_dwes):.4f}")
print(f"  Saved          : submission_0.png ... submission_{len(all_ious)-1}.png")
print(f"{'='*55}")

Output hidden; open in https://colab.research.google.com to view.